# Imports

In [1]:
# Standard Library Imports
from datetime import datetime, time
import random
# Third Party Imports
import numpy as np
# Local Imports
from generator import Generator, Distribution

# Defining Distributions

In [2]:

class MaleDiseasesDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["Disease A", "Disease B", "Disease C", "Disease D"]
        self.probabilities = [0.1, 0.2, 0.3, 0.4]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class FemaleDiseasesDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["Disease F", "Disease G", "Disease H", "Disease I"]
        self.probabilities = [0.4, 0.3, 0.2, 0.1]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class GenderDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["male", "female"]
        self.probabilities = [0.5, 0.5]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class NumberOfPatientsDuringADayDistribution(Distribution):
    def __init__(self):
        super().__init__()

    def sample(self):
        return int(np.random.normal(50, 10))



class ArrivalTime(Distribution):
    def __init__(self):
        super().__init__()
        self.intervals = [(8, 12), (12, 16), (16, 20), (20, 24)]
        self.probabilities = [0.1, 0.3, 0.4, 0.2]

    def sample(self) -> str:
        start, end = random.choices(self.intervals, weights=self.probabilities, k=1)[0]
        sampled = np.random.uniform(start, end)
        h, rem = divmod(sampled * 3600, 3600)
        m, s = divmod(rem, 60)
        return time(int(h), int(m), int(s)).strftime("%H:%M:%S")

class ABDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["A", "B"]
        self.probabilities = [0.5, 0.5]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))
    
# Instantiate the distributions
num_patients_per_day_dist: Distribution = NumberOfPatientsDuringADayDistribution()
male_diseases_dist: Distribution = MaleDiseasesDistribution()
female_diseases_dist: Distribution = FemaleDiseasesDistribution()
gender_dist: Distribution = GenderDistribution()
arrival_time: Distribution = ArrivalTime()
AB_dist: Distribution = ABDistribution()

# Make Distribution Tree

In [ ]:
DistributionTreeNoPatientsPrDay = {
    "number_of_patients": {
        "dist": num_patients_per_day_dist,
        "no_sample_aggregation": lambda x: x,
        "arrival_time": {
            "dist": arrival_time,
            "no_sample_aggregation": lambda x: x[0],
        },
        "gender": {
            "dist": gender_dist,
            "no_sample_aggregation": lambda x: x[0],
            "disease": {
                "dist": {
                    "male": male_diseases_dist,
                    "female": female_diseases_dist
                },
                "no_sample_aggregation": len,
            }
    }
    },

    "other_time_series_data": {
        "dist": num_patients_per_day_dist,
        "no_sample_aggregation": lambda x: x,
        "arrival_time": {
            "dist": arrival_time,
            "no_sample_aggregation": lambda x: x[0],
        },
        "ab": {
            "dist": AB_dist,
            "no_sample_aggregation": lambda x: x[0],
        }
    },
}

# Instantiate and run the generator

In [4]:
generator = Generator(
    distribution_tree=DistributionTreeNoPatientsPrDay, 
    output_format={
        "number_of_patients": ("arrival_time", "gender", "disease"),
        "other_time_series_data": ("arrival_time", "ab")  
    }, 
    output_sorting_function=lambda tup: datetime.strptime(tup[0], "%H:%M:%S").time()
)
output = generator.generate_data()
print(output)

[('08:09:05', 'male', 'Disease D'), ('08:40:49', 'A'), ('09:17:58', 'A'), ('09:27:22', 'B'), ('10:21:05', 'A'), ('11:12:26', 'female', 'Disease G'), ('11:36:17', 'male', 'Disease D'), ('12:14:20', 'B'), ('12:20:06', 'A'), ('12:27:09', 'A'), ('12:28:33', 'A'), ('12:32:07', 'female', 'Disease F'), ('12:32:18', 'B'), ('12:32:41', 'female', 'Disease F'), ('12:35:45', 'B'), ('12:38:11', 'B'), ('12:39:26', 'male', 'Disease B'), ('12:44:30', 'A'), ('12:46:13', 'A'), ('12:49:12', 'male', 'Disease B'), ('13:04:14', 'B'), ('13:18:02', 'female', 'Disease G'), ('13:23:27', 'B'), ('13:27:19', 'male', 'Disease D'), ('14:06:10', 'male', 'Disease B'), ('14:09:06', 'B'), ('14:22:16', 'female', 'Disease G'), ('14:33:30', 'male', 'Disease A'), ('14:34:20', 'B'), ('14:46:17', 'A'), ('15:16:14', 'A'), ('15:22:01', 'B'), ('15:29:35', 'A'), ('15:33:24', 'female', 'Disease I'), ('15:42:50', 'male', 'Disease C'), ('15:44:39', 'female', 'Disease G'), ('15:46:12', 'female', 'Disease G'), ('15:58:44', 'male', 'Di